# Setup

In [1]:
# Import the updated helpers from the project directory. Reloading picks up
# changes to DEFAULT_FAKE_SCALES without rebuilding the histogram workspace.
import sys
import importlib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import helpers as hp

importlib.reload(hp)
print('Using helpers from:', hp.__file__)

# Preserve every Matplotlib/Seaborn setting from the original notebook.
hp.configure_pt_plot_style()

Using helpers from: /home/lessa/UnfoldingBSMttbar/Plots/helpers.py


# Summary plot

In [6]:
# Choose which true-model columns and tested-model rows appear, and their order.
# Model selection uses internal keys; displayed names are edited in the function.
# These lists can contain only a subset of models and need not have equal length.
summary_columns = [
    'Scalar_1500', 'VLF_1500', 'Zprime_3000','Zprime_1pc_3000',  #'SM',
]
summary_rows = [
    'Scalar', 'VLF', 'Zprime_20pc','Zprime', #'SM',
]
summary_lumi = 3000.0
summary_systematic = 0.0
mass_source = 'best_fit_mc'
refit_scale = True
refit_zprime_scale = False
analysis_range = (500, 3600)  # GeV, inclusive lower edge / exclusive upper edge
systematics = [0.00]         # Fractional background uncertainties (0.02 means 2%).
luminosities = np.array([300, 500, 1000, 1500, 2000, 2500, 3000], dtype=float)
workspace_file = 'pT_mass_scan_13_6TeV_hists.pkl'



# Compute only the selected underlying truths. The helper handles SM truth as
# fixed benchmarks and the remaining truths as fitted model hypotheses.
summary_results = hp.make_significance_summary(
    workspace_file=workspace_file, luminosity=summary_lumi,
    systematics=systematics, observable='pT', analysis_range=analysis_range,
    mass_source=mass_source, refit_best_fit_scale=refit_scale,
    refit_zprime_best_fit_scale=refit_zprime_scale,
    truth_keys=summary_columns,
)




# Change these strings to adjust the displayed names, not the selection keys.
truth_labels = {
    'Scalar_1500': 'Scalar', 'VLF_1500': 'VLF',
    'Zprime_3000': r"$Z^\prime$ Broad",
    'Zprime_1pc_3000': r"$Z^\prime$ Narrow",
}
model_labels = {
    'Scalar': 'Scalar', 'VLF': 'VLF',
    'Zprime': r"$Z^\prime$ Narrow",
    'Zprime_20pc': r"$Z^\prime$ Broad",
    'SM': 'SM',
}
# For SM truth, comparison labels are fake-data keys rather than fit models.
fixed_benchmark_to_model = {
    'Scalar_1500': 'Scalar', 'VLF_1500': 'VLF',
    'Zprime_1pc_3000': 'Zprime', 'Zprime_3000': 'Zprime_20pc',
}
row_order = ['Scalar', 'VLF', 'Zprime', 'Zprime_20pc']
column_order = ['Scalar_1500', 'VLF_1500','Zprime_1pc_3000', 'Zprime_3000']
row_order = list(row_order)
column_order = list(column_order)
if not row_order or not column_order:
    raise ValueError('Select at least one tested model and one true model.')
if len(row_order) != len(set(row_order)) or len(column_order) != len(set(column_order)):
    raise ValueError('Repeated row or column keys are not allowed.')
invalid_rows = [key for key in row_order if key not in model_labels]
invalid_columns = [key for key in column_order if key not in summary_results]
if invalid_rows or invalid_columns:
    raise KeyError(f'Unknown rows: {invalid_rows}; unknown truth columns: {invalid_columns}')

summary_col = "Z_sh_eps_00"
# Keep internal keys while assembling the matrix; map only display labels.
df_significance = pd.DataFrame(index=row_order, columns=column_order, dtype=float)
for fake_key in column_order:
    df = summary_results[fake_key]
    if summary_col not in df.columns:
        raise KeyError(f'Missing significance column {summary_col!r} for {fake_key!r}')
    for _, result_row in df.iterrows():
        tested_key = str(result_row['pair']).split(' vs ', 1)[-1]
        model_key = (fixed_benchmark_to_model.get(tested_key, tested_key)
                        if fake_key == 'SM' else tested_key)
        if model_key in df_significance.index:
            df_significance.loc[model_key, fake_key] = float(result_row[summary_col])

df_significance.index = [model_labels[key] for key in row_order]
df_significance.columns = [truth_labels.get(key, key) for key in column_order]




Fitting Scalar_1500 once at 3000 fb^-1 (0% systematics).

Fitting observable: pT
Analysis range: 500 <= pT < 3600 GeV
Systematic error: 0.0%
VLF         : selected mass = 1744.000 GeV | parabolic estimate = 1744.502 GeV | scale = 4.5628719 | chi2(500/fb) = 0.845848 (31 bins) | source = best_fit_mc
Scalar      : selected mass = 1510.000 GeV | parabolic estimate = 1510.184 GeV | scale = 7.577717 | chi2(500/fb) = 0.00877195 (31 bins) | source = best_fit_mc
Zprime      : selected mass = 2500.000 GeV | parabolic estimate = 2500.067 GeV | scale = -0.33389007 | chi2(500/fb) = 23.2898 (31 bins) | source = best_fit_mc
Zprime_20pc : selected mass = 2882.000 GeV | parabolic estimate = 2882.792 GeV | scale = -3.837104 | chi2(500/fb) = 10.4665 (31 bins) | source = best_fit_mc
FakeData    : nominal mass = 1500.0 GeV | scale = 7.5
Synthetic signal yield in fit window: 18126.74 events
Fitting VLF_1500 once at 3000 fb^-1 (0% systematics).

Fitting observable: pT
Analysis range: 500 <= pT < 3600 GeV
Sys

In [ ]:

plt.figure(figsize=(11, 7))
ax = sns.heatmap(
    df_significance, annot=True, fmt='.1f', cmap='Blues',
    cbar_kws={'label': 'Asimov significance (Z)'},
    linewidths=1, linecolor='black',
    annot_kws={'size': 25},
)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=25)
cbar.set_label(r'Significance ($Z_A$)', fontsize=27)

plt.yticks(rotation=0)
plt.tick_params(axis='both', which='major', labelsize=23)
plt.title(r'$\mathcal{L}=3\,\mathrm{ab}^{-1}$', fontsize=27)
plt.xlabel(r'True Hypothesis ($H_1$)', labelpad = 10, fontsize=26)
plt.ylabel(r'Tested hypothesis ($H_0$)', labelpad = 10, fontsize=26)
plt.tight_layout()
plt.savefig('./../paper_plots/summary_plot_significance.png', bbox_inches='tight', dpi=300)
plt.show()
